---
layout: post
courses: { csa: {week: 19} }
toc: true
codemirror: true
title: 2018 FRQ Question 1
year: 2018
frq_number: 1
unit: Frog Simulation
category: Simulation with Loops
description: "AP CSA 2018 FRQ #1 - Frog Simulation"
permalink: /csa/frqs/2018/1
author: Nora, Spencer, Xavier
---

## Question Overview

This question involves simulating a frog attempting to reach a goal distance by hopping. You will implement two methods for the `FrogSimulation` class.

<p align="center">
  <img src="/images/gamify/frog.gif" alt="Frog animation" width="300" />
</p>

## FrogSimulation class Specification

- `goalDistance` = (target position)

- maxHops` = (maximum hops allowed)

- `hopDistance()` = A private method that returns the distance of a single hop (can be positive, negative, or zero)

- `simulate()` Returns **true** if the frog reaches the goal without going negative; **false** otherwise

- `runSimulations(int num)` Runs **num** simulations and returns the proportion of successful attempts as a **double**

### Key Insights

- The frog starts at position 0
- After each hop, check if position < 0 (fail immediately) or position >= goalDistance (succeed immediately)
- For `runSimulations`, use `(double) successes / num` to avoid integer division

In [ ]:
// CODE_RUNNER: FrogSimulation

class FrogSimulation {
    private int goalDistance;
    private int maxHops;

    public FrogSimulation(int dist, int numHops) {
        goalDistance = dist;
        maxHops = numHops;
    }

    private int hopDistance() {
        return (int) (Math.random() * 11) - 2; // -2 to 8
    }

    public boolean simulate() {
        int position = 0;

        for (int i = 0; i < maxHops; i++) {
            position += hopDistance();

            if (position < 0) {
                return false;
            }

            if (position >= goalDistance) {
                return true;
            }
        }

        return false;
    }

    public double runSimulations(int num) {
        int successes = 0;

        for (int i = 0; i < num; i++) {
            if (simulate()) {
                successes++;
            }
        }

        return (double) successes / num;
    }
}

class Main {
    public static void main(String[] args) {
        FrogSimulation sim = new FrogSimulation(24, 5);

        System.out.println("Single simulation: " + sim.simulate());
        System.out.println("Proportion successful in 1000 runs: " + sim.runSimulations(1000));
    }
}

Main.main(null);

---

### Popcorn Hack #1

> **Question:** In `simulate()`, what should happen immediately if the frog’s position becomes negative after a hop?

<details>
<summary>Show answer</summary>

`simulate()` should return `false` immediately because the run fails as soon as position is below 0.
</details>

----

## Example 1: Successful Simulation

**Given:** `goalDistance = 24`, `maxHops = 5`, Hop sequence: 5, 7, -2, 8, 6

| Hop | Distance | Position | Negative? | Goal? | Result |
|-----|----------|----------|-----------|-------|--------|
| 1 | 5 | 5 | No | No | Continue |
| 2 | 7 | 12 | No | No | Continue |
| 3 | -2 | 10 | No | No | Continue |
| 4 | 8 | 18 | No | No | Continue |
| 5 | 6 | 24 | No | Yes (24 >= 24) | **Return true** |

**What you should see when you run the code:**
- Single simulation: true or false (varies due to randomness)
- Proportion: approximately 0.3-0.5 (varies)

---

### Popcorn Hack #2

> **Question:** In Example 1, why does the method return `true` on hop 5?

<details>
<summary>Show answer</summary>

Because the position reaches 24, and `24 >= goalDistance`, so the goal condition is satisfied.
</details>

---

## Example 2: Frog Goes Negative (Failure)

**Given:** `goalDistance = 24`, `maxHops = 5`, Hop sequence: 6, -7, ...

| Hop | Distance | Position | Negative? | Goal? | Result |
|-----|----------|----------|-----------|-------|--------|
| 1 | 6 | 6 | No | No | Continue |
| 2 | -7 | -1 | **Yes** | N/A | **Return false** |

**Key Point:** The simulation ends immediately when the frog goes negative. Remaining hops are not executed.

---

### Popcorn Hack #3

> **Question:** In Example 2, after the frog reaches `-1` on hop 2, should the simulation continue to hop 3?

<details>
<summary>Show answer</summary>

No. The simulation stops immediately and returns `false` when position is negative.
</details>

---

## Solution Explanation

Here is the complete implementation that satisfies all 9 points on the rubric.

### Part (a): simulate() - 5 Points

In [ ]:
// CODE_RUNNER: Solution Example Part (a) - simulate()
class FrogSimulation {
    private int goalDistance;
    private int maxHops;
    private int[] testHops;
    private int hopIndex;

    public FrogSimulation(int dist, int numHops, int[] hops) {
        goalDistance = dist;
        maxHops = numHops;
        testHops = hops;
        hopIndex = 0;
    }

    private int hopDistance() {
        if (hopIndex < testHops.length) {
            return testHops[hopIndex++];
        }
        return 0;
    }

    public boolean simulate() {
        int position = 0;

        for (int i = 0; i < maxHops; i++) {
            position += hopDistance();

            if (position < 0) {
                return false;
            }

            if (position >= goalDistance) {
                return true;
            }
        }

        return false;
    }
}

class Main {
    public static void main(String[] args) {
        FrogSimulation successCase = new FrogSimulation(24, 5, new int[]{5, 7, -2, 8, 6});
        FrogSimulation failCase = new FrogSimulation(24, 5, new int[]{6, -7, 8, 8, 8});
        System.out.println("Part (a) expected true: " + successCase.simulate());
        System.out.println("Part (a) expected false: " + failCase.simulate());
    }
}

Main.main(null);

### How It Works
- Initialize position to 0
- Loop exactly `maxHops` times
- Update position with `hopDistance()`
- Check if position < 0 and return `false` immediately
- Check if position >= goalDistance and return `true` immediately
- Return `false` after loop if goal not reached


### Part (b): runSimulations(int num) - 4 Points

In [ ]:
// CODE_RUNNER: Solution Example Part (b) - runSimulations(int num)
class FrogSimulation {
    private int goalDistance;
    private int maxHops;
    private int[][] trials;
    private int trialIndex;
    private int hopIndex;

    public FrogSimulation(int dist, int numHops, int[][] allTrials) {
        goalDistance = dist;
        maxHops = numHops;
        trials = allTrials;
        trialIndex = 0;
        hopIndex = 0;
    }

    private int hopDistance() {
        if (trialIndex >= trials.length) return 0;
        int[] currentTrial = trials[trialIndex];
        if (hopIndex < currentTrial.length) return currentTrial[hopIndex++];
        return 0;
    }

    public boolean simulate() {
        if (trialIndex >= trials.length) return false;

        int position = 0;
        hopIndex = 0;

        for (int i = 0; i < maxHops; i++) {
            position += hopDistance();
            if (position < 0) {
                trialIndex++;
                return false;
            }
            if (position >= goalDistance) {
                trialIndex++;
                return true;
            }
        }

        trialIndex++;
        return false;
    }

    public double runSimulations(int num) {
        int successes = 0;
        for (int i = 0; i < num; i++) {
            if (simulate()) successes++;
        }
        return (double) successes / num;
    }
}

class Main {
    public static void main(String[] args) {
        int[][] sampleTrials = {
            {5, 7, -2, 8, 6},
            {6, -7, 8, 8, 8},
            {8, 8, 8, 0, 0},
            {1, 1, 1, 1, 1}
        };

        FrogSimulation sim = new FrogSimulation(24, 5, sampleTrials);
        System.out.println("Part (b) expected 0.5: " + sim.runSimulations(4));
    }
}

Main.main(null);

### How It Works

1. `simulate()` tracks the frog's position starting from 0, updating after each hop
2. The method returns immediately when the frog goes negative or reaches the goal
3. `runSimulations(int num)` calls `simulate()` multiple times and calculates the success rate
4. Casting to `double` is essential to get a decimal proportion instead of 0

---

## Scoring Guidelines

### Points Breakdown (9 points total)

#### Part (a): simulate() (5 points)
- **1 point:** Initializes position variable to 0
- **1 point:** Loops exactly `maxHops` times
- **1 point:** Calls `hopDistance()` and updates position
- **1 point:** Checks if position < 0 inside loop and returns `false` immediately
- **1 point:** Checks if position >= goalDistance inside loop and returns `true`; returns `false` after loop

#### Part (b): runSimulations(int num) (4 points)
- **1 point:** Initializes counter to 0
- **1 point:** Loops `num` times
- **1 point:** Calls `simulate()` and increments counter when true
- **1 point:** Returns `(double) successes / num`

### Common Mistakes to Avoid
- Checking conditions outside the loop instead of inside
- Using `==` instead of `>=` for goal check
- Using `position <= 0` instead of `position < 0`
- Not casting to double: `successes / num` gives 0 instead of proportion
- Not returning immediately when conditions are met